# 10 · Error analysis

Builds the error-analysis table required by the report:

| meeting_id | component | input | prediction | reference | error_type | comment |

Sources: `outputs/metrics/predictions_<split>.json`, `outputs/summaries/`,
`outputs/minutes/*.json`. Automated detectors flag candidate errors; every row is
meant to be **read and annotated by a human** — detectors only prioritise.

In [ ]:
print('[1/4] Load predictions and references')
import sys; sys.path.insert(0, '.')
import json
from pathlib import Path
from src.config import load_config, resolve_path
cfg = load_config()
pred_path = resolve_path(cfg, 'metrics') / 'predictions_val.json'
if not pred_path.exists():
    print('Run !python scripts/evaluate.py --limit 5 first.')
    data = {}
else:
    data = json.loads(pred_path.read_text(encoding='utf-8'))
    print('meetings:', len(data.get('predictions', {})))

In [ ]:
print('[2/4] Detect candidate summary errors (rules, conservative)')
from src.evaluation import rouge_scores
rows = []
for mid, preds in data.get('predictions', {}).items():
    mp = resolve_path(cfg, 'data_processed') / f'{mid}.json'
    if not mp.exists():
        continue
    ref = json.loads(mp.read_text(encoding='utf-8')).get('reference_summary', '')
    for system, pred in preds.items():
        if not pred or not ref:
            continue
        r = rouge_scores([pred], [ref])
        if r['rouge1'] is not None and r['rouge1'] < 0.20:
            rows.append({'meeting_id': mid, 'component': system, 'input': '(transcript)',
                         'prediction': pred[:300], 'reference': ref[:300],
                         'error_type': 'low_overlap', 'comment': 'ROUGE-1 < 0.20; check for missing decisions'})
        if len(pred.split()) and ref.lower().split() and pred.count('.') >= 6:
            rows.append({'meeting_id': mid, 'component': system, 'input': '(transcript)',
                         'prediction': pred[:300], 'reference': ref[:300],
                         'error_type': 'length_mismatch' if abs(len(pred.split()) - len(ref.split())) > 120 else 'ok',
                         'comment': 'review length'})
print('candidate rows:', len(rows))

In [ ]:
print('[3/4] Save the table for manual annotation')
from src.evaluation import write_results
import pandas as pd
out = Path('outputs/metrics/error_analysis.csv')
df = pd.DataFrame(rows, columns=['meeting_id', 'component', 'input', 'prediction',
                                 'reference', 'error_type', 'comment'])
df.to_csv(out, index=False, encoding='utf-8')
print('->', out, f'({len(df)} rows)')

In [ ]:
print('[4/4] Manual review checklist')
print('For each row classify the real failure mode:')
print('  summary_misses_decision | summary_hallucinates | action_missed | wrong_owner |')
print('  wrong_deadline | asr_error | classifier_confusion | chunk_context_loss')
print('and record findings in reports/experiment_results.md.')